<a href="https://colab.research.google.com/github/67160330/Project/blob/main/Workshop_67160330.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 EXTRACT (การอ่านข้อมูลดิบ)

In [10]:
import pandas as pd
import sqlite3

df = pd.read_csv('raw_ecommerce_data.csv')

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


# Phase2A TRANSFROM (สร้างDimension Table)

In [11]:
# สร้าง dim_customer
dim_customer = df[['Customer_Name', 'Email']].drop_duplicates()

# จัดการ Missing Values
dim_customer = dim_customer.dropna(subset=['Customer_Name', 'Email'])

# สร้าง Surrogate Key (customer_id)
dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

# จัดเรียงคอลัมน์ให้ PK อยู่หน้าสุด
dim_customer = dim_customer[['customer_id', 'Customer_Name', 'Email']]

# Phase2B สร้าง Fact Table

In [12]:
# นำ customer_id กลับไปใส่ใน fact table ผ่านการ Join
fact_sales = pd.merge(df, dim_customer,
                      on=['Customer_Name', 'Email'],
                      how='left')

# ลบคอลัมน์ Text ทิ้ง เหลือไว้เพียง Foreign Key
fact_sales = fact_sales.drop(columns=['Customer_Name', 'Email'])

# ทำซ้ำกระบวนการนี้กับ Product และ Time Dimensions

# Phase3A LOAD (ตั้งค่า SQLite Warehouse)

In [16]:
# สร้าง Connection ไปที่ไฟล์ .db
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

# สร้างตาราง Dimension พร้อมกำหนด Primary Key
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_id INTEGER PRIMARY KEY,
        Customer_Name TEXT,
        Email TEXT
    )
''')

conn.commit()

# Phase3B บังคับใช้ Star Schema Relationship

In [18]:
#-- เปิดใช้งานการตรวจสอบ Foreign Key ใน SQLite
cursor.execute('PRAGMA foreign_keys = ON;')

cursor.execute('''
CREATE TABLE IF NOT EXISTS fact_sales (
    transaction_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (product_id) REFERENCES dim_product(product_id)
);
''')
conn.commit()

# Phase3C ผลักข้อมูลลง Warehouse

In [19]:
# โหลดข้อมูล Dimension
dim_customer.to_sql('dim_customer', con=conn,
                    if_exists='replace', index=False)

# โหลดข้อมูล Fact
fact_sales.to_sql('fact_sales', con=conn,
                  if_exists='replace', index=False)

print('ETL Pipeline ran successfully!')

ETL Pipeline ran successfully!


# ทดสอบคิวรีจาก Warehouse

In [20]:
sql_query = """
SELECT
    c.Customer_Name,
    SUM(f.amount) as Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.Customer_Name
ORDER BY Total_Spend DESC
LIMIT 3;
"""

df_result = pd.read_sql_query(sql_query, conn)
print(df_result)

conn.close()

  Customer_Name  Total_Spend
0     Narin Dee      37597.5
1    Alice Wong      25009.5
2      Krit Som      23976.0
